In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter

from utils import *

In [5]:
folder = 'temp/'
files = sorted(glob.glob(folder + '*.fits'))
files

['temp/phi-fdt-flat_20260425T230003_V202607091333C_0664250100.fits',
 'temp/phi-fdt-ghost_20260425T230003_V202607091333C_0664250100.fits']

In [6]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
flat_file = files[0]
ghost_file = files[1]

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

ghost = demodulate(ghost)
flat_ = demodulate(flat)

In [8]:
plt.figure(figsize=(10,10))
plt.imshow(flat_[2], vmin=-5e-3, vmax=5e-3)

In [9]:
folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25/solo_L1_phi-fdt-ilam_20260425T230003_V202605051433C_0664250100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25/solo_L1_phi-fdt-ilam_20260425T230404_V202605051633C_0664250125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25/solo_L1_phi-fdt-ilam_20260425T230804_V202605051733C_0664250150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25/solo_L1_phi-fdt-ilam_20260425T231204_V202605121034C_0664250175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25/solo_L1_phi-fdt-ilam_20260425T231604_V202605121135C_0664250200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25/solo_L1_phi-fdt-ilam_20260425T232004_V202605121135C_0664250225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026-04-25/solo_L1_phi-fdt-ilam_20260425T232404_V202605201333C_0664250250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2026

In [10]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

cpos = header['CONTPOS'] - 1
wv = read_wavelengths(header)
#data = data.reshape(6,4,2048,2048)[cpos]
data = calc_continuum(data, wv, continuum=cpos)

data -= dark * 0.4
data /= flat
data = realign(data)
data = demodulate(data)

xr, yr = reflection_point_predict(header)
reflection = reflect(gaussian_filter(data[0], 8), xr, yr)

In [11]:
plt.figure(figsize=(10,10))
plt.imshow(data[0])
plt.tight_layout()

In [14]:
plt.figure(figsize=(10,10))
plt.imshow(data[3] - ghost[3] * reflection, vmin=-30, vmax=30)
plt.tight_layout()